# Imports

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    import pulp as pl
except ImportError:
    import os
    os.system("pip install pulp")
    import pulp as pl

from IPython.display import clear_output

# Input Data

In [21]:
budget = 100000
risk = 23
expected_return = 0
stocks = ["AAPL","^GSPC"]
days = 365

# VaR 

In [22]:

import numpy as np

def VaR(ticker, p=0.95):
    try:
        import yfinance as yf
    except ImportError:
        import os
        os.system("pip install yfinance")
        import yfinance as yf

    try:
        from scipy.stats import norm
    except ImportError:
        import os
        os.system("pip install scipy")
        from scipy.stats import norm

    # fetch all historical data for the ticker
    data = yf.download(ticker, progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")


    returns = (data['Open']-data['Close'])/data['Open']
    returns = returns[returns.columns[0]]
    E = returns.mean()
    sigma = returns.std()
    Z_p = norm.ppf(1 - p)
    VaR = (Z_p * sigma) - E
    return VaR 

# RoI

In [23]:

import numpy as np

def RoI(ticker, p=0.95,days=365):
    try:
        import yfinance as yf
    except ImportError:
        import os
        os.system("pip install yfinance")
        import yfinance as yf

    try:
        from scipy.stats import norm
    except ImportError:
        import os
        os.system("pip install scipy")
        from scipy.stats import norm

    # fetch all historical data for the ticker
    data = yf.download(ticker,start="1900-01-01" ,progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")

    returns = (data['Open']-data['Close'].shift(days))/data['Open']
    returns = returns[returns.columns[0]]
    E = returns.mean()
    return E 

# Optimization model

In [24]:
def optimize(budget,stocks,risk,expected_return,days,VaRs,RoIs):

    # define problem
    z = pl.LpProblem("Portfolio_Optimization", pl.LpMinimize)

    # define decision variables
    inv = pl.LpVariable.dicts("Stocks", stocks, lowBound=0, cat='Continuous')

    #define continuity variable
    total_spent = pl.LpVariable("Total_Investment", lowBound=0, cat='Continuous')
    for t in stocks:
        z+= inv[t]>=0

    # risk slack variable
    s1 = pl.LpVariable("Risk_Slack", lowBound=0, cat='Continuous')
    s2 = pl.LpVariable("Return_Slack", lowBound=0, cat='Continuous')


    # define objective function
    z += s1 + s2, "Minimize_Slack_Variables"

    # continuity constraints
    z+= pl.lpSum([inv[ticker] for ticker in inv]) == total_spent
    z+= total_spent <= budget
    z+= pl.lpSum([inv[ticker] for ticker in inv]) >= 1
    
    # define constraints
    z += pl.lpSum([inv[ticker] for ticker in inv]) <= budget, "Budget_Constraint"
    z += pl.lpSum([inv[ticker] * VaRs[ticker] for ticker in inv]) == total_spent * risk + s1, "Risk_Constraint"
    z += pl.lpSum([inv[ticker] * RoIs[ticker] for ticker in inv]) == total_spent * expected_return - s2, "Return_Constraint"

    # solve problem
    z.solve()
    clear_output()

    return z


# RHS Bounds

In [ ]:
def get_rhs_bounds(budget,stocks,risk,expected_return,days,VaRs,RoIs,c_name):
    p = {
        "budget": budget,
        "risk": risk,
        "expected_return": expected_return
    }

    z = optimize(p["budget"],
                 stocks,
                 p["risk"],
                 p["expected_return"],
                 days,VaRs,RoIs)
    
    # Current values
    current_obj = z.objective.value()
    current_rhs = z.constraints[c_name].constant


    tol = 1e-5

    # find upper bound (binary search)
    l_bound = current_rhs
    u_bound = current_rhs * 1e6

    while u_bound - l_bound > tol:
        p["c_name"] = (l_bound + u_bound) / 2

        zval = optimize(p["budget"],
                        stocks,
                        p["risk"],
                        p["expected_return"],
                        days, VaRs, RoIs).objective.value()
        if zval > current_obj + tol:
            l_bound = p["c_name"]
        else:
            u_bound = p["c_name"]

    rhs_upper = (l_bound + u_bound) / 2

    # find lower bound (binary search)
    l_bound = current_rhs / 1e6
    u_bound = current_rhs

    while u_bound - l_bound > tol:
        p["c_name"] = (l_bound + u_bound) / 2

        zval = optimize(p["budget"],
                        stocks,
                        p["risk"],
                        p["expected_return"],
                        days, VaRs, RoIs).objective.value()
        if zval > current_obj + tol:
            u_bound = p["c_name"]
        else:
            l_bound = p["c_name"]

    rhs_lower = (l_bound + u_bound) / 2

    return rhs_lower, rhs_upper

# Report

In [26]:

VaRs = {ticker: VaR(ticker) for ticker in stocks}
RoIs = {ticker: RoI(ticker, days=days) for ticker in stocks}


z = optimize(budget,stocks,risk,expected_return,days,VaRs,RoIs)
varvals = {v.name: v.varValue for v in z.variables()}

# print inputs
print("Inputs:")
print("#"*20)
print("Budget:", budget)
print("Stocks:", stocks)
print("Risk (VaR):", risk)
print("Expected Return:", expected_return)


print(z.objective.value())

# print results
for v in z.variables():
    print(f"{v.name}: {v.varValue}")
print(f"Objective value: {pl.value(z.objective)}")

print("\n"*3)

print("Total Expected Return:",end=" ")
ter = sum([varvals[f'Stocks_{ticker}'] * RoIs[ticker] for ticker in stocks])
print(ter)
print("Total Risk (VaR):",end=" ")
trisk = sum([varvals[f'Stocks_{ticker}'] * VaRs[ticker] for ticker in stocks])
print(trisk)

Inputs:
####################
Budget: 100000
Stocks: ['AAPL', '^GSPC']
Risk (VaR): 23
Expected Return: 0
0.0
Return_Slack: 0.0
Risk_Slack: 0.0
Stocks_AAPL: -4427.2751
Stocks_^GSPC: 4428.2751
Total_Investment: 1.0
Objective value: 0.0




Total Expected Return: -236.389918726834
Total Risk (VaR): 22.999999965933966
